# CLIP Image Embeddings

Embed the 510-image evaluation set with **OpenAI CLIP ViT-B/32** using `open-clip-torch`.

- First run downloads weights (~350 MB) and caches in `~/.cache/huggingface/`
- CPU-only (X1 Carbon); uses all available threads
- Saves embeddings as a numpy `.npz` file for downstream use

In [ ]:
import torch
import open_clip
import numpy as np
from PIL import Image
from pathlib import Path

print('torch:', torch.__version__)
print('open_clip:', open_clip.__version__)
print('CPU threads:', torch.get_num_threads())

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
IMG_DIR  = Path('../vc_genome_output_full/images')   # adjust if needed
OUT_FILE = Path('clip_embeddings_vitb32.npz')
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'openai'

torch.set_num_threads(8)  # max out CPU cores on X1 Carbon

In [ ]:
# ── Load model ──────────────────────────────────────────────────────────────
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model.eval()
print(f'Model loaded: {MODEL_NAME} ({PRETRAINED})')
print(f'Embedding dim: {model.visual.output_dim}')

In [ ]:
# ── Embed images ─────────────────────────────────────────────────────────────
image_paths = sorted(IMG_DIR.glob('*.png')) + sorted(IMG_DIR.glob('*.jpg'))
print(f'Found {len(image_paths)} images in {IMG_DIR}')

names, vecs = [], []

with torch.no_grad():
    for i, p in enumerate(image_paths):
        img = preprocess(Image.open(p).convert('RGB')).unsqueeze(0)
        emb = model.encode_image(img)
        names.append(p.name)
        vecs.append(emb.squeeze().numpy())
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(image_paths)}')

embeddings = np.stack(vecs)  # shape: (N, 512)
print(f'Embedding matrix: {embeddings.shape}')

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
np.savez(OUT_FILE, names=np.array(names), embeddings=embeddings)
print(f'Saved -> {OUT_FILE}  ({embeddings.shape[0]} images, dim={embeddings.shape[1]})')

In [ ]:
# ── Quick sanity check ───────────────────────────────────────────────────────
data = np.load(OUT_FILE, allow_pickle=True)
print('names sample:', data['names'][:5])
print('embeddings shape:', data['embeddings'].shape)

# Cosine similarity between first two images
from numpy.linalg import norm
e = data['embeddings']
e_norm = e / norm(e, axis=1, keepdims=True)
sim = e_norm[0] @ e_norm[1]
print(f'Cosine sim between [{data["names"][0]}] and [{data["names"][1]}]: {sim:.4f}')

## VC Prediction from CLIP Embeddings

Predict NormalizedVC from CLIP embeddings using:
1. **Ridge regression** (L2-regularised linear) — baseline
2. **SVR (RBF kernel)** — nonlinear comparison
3. **k-NN regression** — sanity check: do visually similar images share similar VC?

Evaluation: 5-fold cross-validation, reporting **CCC**, **R²** (identity-line), and **MAE** to match the LLM appendix metrics.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import norm
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr

# ── Lin's CCC ────────────────────────────────────────────────────────────────
def concordance_cc(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    my, mh = y.mean(), yhat.mean()
    vy, vh = y.var(), yhat.var()
    cov = ((y - my) * (yhat - mh)).mean()
    denom = vy + vh + (my - mh) ** 2
    return 2 * cov / denom if denom > 0 else np.nan

# ── Load embeddings ───────────────────────────────────────────────────────────
OUT_FILE = Path('clip_embeddings_vitb32.npz')
data = np.load(OUT_FILE, allow_pickle=True)
emb_names  = list(data['names'])
embeddings = data['embeddings'].astype(np.float32)

# L2-normalise (standard for CLIP)
embeddings = embeddings / norm(embeddings, axis=1, keepdims=True)
print(f'Embeddings: {embeddings.shape}')

# ── Load GT ──────────────────────────────────────────────────────────────────
base = Path('..') 
gt = pd.read_csv(base / 'phrase_reduction_v2' / 'image_compiled_phrases.csv',
                 usecols=['imageName', 'NormalizedVC']).dropna()
gt = gt.drop_duplicates('imageName').set_index('imageName')

# Align: keep only images present in both
common = [n for n in emb_names if n in gt.index]
idx    = [emb_names.index(n) for n in common]
X      = embeddings[idx]
y      = gt.loc[common, 'NormalizedVC'].values
print(f'Aligned: {len(common)} images with GT')


In [ ]:
# ── 5-fold cross-validation ──────────────────────────────────────────────────
ANCHOR_IMAGES = {'VisC.503.6.png', 'InfoVisJ.619.17.png', 'InfoVisJ.1149.6(1).png'}
mask = np.array([n not in ANCHOR_IMAGES for n in common])
X_cv, y_cv = X[mask], y[mask]
names_cv   = [n for n, m in zip(common, mask) if m]
print(f'CV set (anchors excluded): {len(y_cv)} images')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge':   RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]),
    'SVR-RBF': SVR(kernel='rbf', C=10, epsilon=0.05),
    'k-NN(k=7)': KNeighborsRegressor(n_neighbors=7, metric='cosine'),
}

results = []
for name, clf in models.items():
    cccs, r2s, maes = [], [], []
    for train_idx, val_idx in kf.split(X_cv):
        Xtr, Xval = X_cv[train_idx], X_cv[val_idx]
        ytr, yval = y_cv[train_idx], y_cv[val_idx]

        # Scale features for SVR; Ridge/kNN are scale-invariant on L2-normed data
        if name == 'SVR-RBF':
            sc = StandardScaler()
            Xtr  = sc.fit_transform(Xtr)
            Xval = sc.transform(Xval)

        clf.fit(Xtr, ytr)
        ypred = clf.predict(Xval)

        cccs.append(concordance_cc(yval, ypred))
        r2s.append(r2_score(yval, ypred))
        maes.append(mean_absolute_error(yval, ypred))

    results.append({
        'Model':  name,
        'CCC':    round(np.mean(cccs), 4),
        'R²':     round(np.mean(r2s),  4),
        'MAE':    round(np.mean(maes), 4),
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

# Scatter: Ridge predicted vs GT (train on full, plot for reference)
sc = StandardScaler()
ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100])
ridge.fit(X_cv, y_cv)
ypred_ridge = ridge.predict(X_cv)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_cv, ypred_ridge, alpha=0.4, s=15)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('GT NormalizedVC')
ax.set_ylabel('Predicted (Ridge, train set)')
ccc_val = concordance_cc(y_cv, ypred_ridge)
r2_val  = r2_score(y_cv, ypred_ridge)
ax.set_title(f'Ridge (train) — CCC={ccc_val:.3f}, R²={r2_val:.3f}')
plt.tight_layout()
plt.savefig('clip_ridge_scatter.pdf', bbox_inches='tight')
plt.show()
print('Note: this is train-set fit — use CV metrics above for honest evaluation')
